In [1]:
# Cell 1 — Imports + small utilities
import os
import glob
import json
from pathlib import Path
from tqdm import tqdm

import numpy as np
import rasterio

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

from terratorch import BACKBONE_REGISTRY
import torch.nn as nn

# For plotting (visualization cell)
import matplotlib.pyplot as plt


In [2]:
# Cell 2 — Paths (change these to your paths if different)
TRAIN_DATA_PATH = r"D:\TerraMind_Dataset\train"
VAL_DATA_PATH   = r"D:\TerraMind_Dataset\val"

# Where to store the mapping and outputs
LABEL_MAP_FILE = "label_mapping.json"
OUTPUT_DIR = Path("./terramind_training_output")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print("Train path:", TRAIN_DATA_PATH)
print("Val   path:", VAL_DATA_PATH)
print("Output dir:", OUTPUT_DIR.absolute())


Train path: D:\TerraMind_Dataset\train
Val   path: D:\TerraMind_Dataset\val
Output dir: c:\Users\sathv\OneDrive\Desktop\Sathvik education\5th Semester\PE\terramind_training_output


In [ ]:
# Cell 4 — Configuration (pixel-wise regression)
CONFIG = {
    "image_size": 224,
    "batch_size": 1,
    "learning_rate": 1e-4,
    "max_epochs": 1,
    "precision": 32,

    # TerraMind backbone
    "model_name": "terramind_v1_base",
    "modalities": ["S2L2A"],
    "bands": None,

    # output channels = spectral bands
    "num_output_channels": 12
}

print("CONFIG:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Cell 5 — Dataset for pixel-wise regression
class TerraMindDataset(Dataset):
    def __init__(self, data_path, image_size=224):
        self.data_path = Path(data_path)
        self.images_path = self.data_path / "images"
        self.targets_path = self.data_path / "targets"
        self.image_size = image_size

        self.samples = sorted([p.stem for p in self.images_path.glob("*.tif")] +
                              [p.stem for p in self.images_path.glob("*.tiff")])

        print(f"✓ Found {len(self.samples)} samples in {data_path}")

    def __len__(self):
        return len(self.samples)

    def _load_and_process(self, path):
        with rasterio.open(str(path)) as src:
            img = src.read()

        if img.shape[0] > 12:
            img = img[:12]
        if img.shape[0] < 12:
            raise ValueError("Expected 12 bands.")

        img = img.astype(np.float32)
        if img.max() > 2.0:
            img /= 10000.0
        img = np.clip(img, 0, 1)

        img = torch.from_numpy(img).unsqueeze(0)
        img = F.interpolate(
            img,
            size=(self.image_size, self.image_size),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        return img

    def __getitem__(self, idx):
        sample_id = self.samples[idx]

        img_path = self.images_path / f"{sample_id}.tif"
        if not img_path.exists():
            img_path = self.images_path / f"{sample_id}.tiff"

        tgt_path = self.targets_path / f"{sample_id}.tif"
        if not tgt_path.exists():
            tgt_path = self.targets_path / f"{sample_id}.tiff"

        X = self._load_and_process(img_path)
        Y = self._load_and_process(tgt_path)

        data = {"S2L2A": X}
        target = Y

        return data, target

In [ ]:
# Cell 6 — Create datasets & dataloaders
train_dataset = TerraMindDataset(
    TRAIN_DATA_PATH,
    image_size=CONFIG["image_size"]
)

val_dataset = TerraMindDataset(
    VAL_DATA_PATH,
    image_size=CONFIG["image_size"]
)

# Windows + rasterio
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("DataLoaders ready!")
print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))

# Sanity check
batch = next(iter(train_loader))
data_sample, target_sample = batch

print("Data keys:", list(data_sample.keys()))
for k, v in data_sample.items():
    print(f"  {k}: {v.shape}, range=[{v.min():.3f}, {v.max():.3f}]")

print("Target shape:", target_sample.shape)
print(
    "Target range:",
    f"[{target_sample.min().item():.3f}, {target_sample.max().item():.3f}]"
)

assert target_sample.dtype == torch.float32

In [ ]:
# Cell 7 — Model definition (pixel-wise regression)
class TerraMindFineTuner(pl.LightningModule):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.backbone = BACKBONE_REGISTRY.build(
            config["model_name"],
            pretrained=True,
            modalities=config["modalities"],
            bands=config["bands"],
            merge_method="mean"
        )

        # Decoder outputs 12 bands (Sentinel-2 reflectance)
        self.decoder = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 12, kernel_size=1)
        )

        self.loss_fn = nn.L1Loss()

    def forward(self, data):
        out = self.backbone(data)

        if isinstance(out, list):
            tokens = out[0]
        else:
            tokens = out

        B, N, C = tokens.shape
        H = W = int(N ** 0.5)

        feat = tokens.transpose(1, 2).reshape(B, C, H, W)
        feat = F.interpolate(
            feat,
            size=(self.config["image_size"], self.config["image_size"]),
            mode="bilinear",
            align_corners=False
        )

        return self.decoder(feat)

    def training_step(self, batch, batch_idx):
        data, y = batch
        y_hat = self(data)

        loss = self.loss_fn(y_hat, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        data, y = batch
        y_hat = self(data)

        loss = self.loss_fn(y_hat, y)
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.config["learning_rate"])

In [ ]:
# Cell 8 — Trainer, callbacks, logger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

output_dir = OUTPUT_DIR

checkpoint_callback = ModelCheckpoint(
    dirpath=output_dir / "checkpoints",
    filename="terramind-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss",
    mode="min",
    save_top_k=3,
    save_last=True
)

early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=8,
    mode="min"
)

logger = TensorBoardLogger(
    save_dir=output_dir / "logs",
    name="terramind_finetuning"
)

print(f"Outputs will be saved to: {output_dir.absolute()}")

model = TerraMindFineTuner(CONFIG)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model params: total={total_params:,}, trainable={trainable_params:,}")

trainer = pl.Trainer(
    max_epochs=CONFIG["max_epochs"],
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision=16 if torch.cuda.is_available() else 32,
    callbacks=[checkpoint_callback, early_stop_callback],
    logger=logger,
    log_every_n_steps=1,
    gradient_clip_val=1.0,
    accumulate_grad_batches=2,
    enable_progress_bar=True,
    enable_model_summary=True
)

print("Trainer initialized.")

Outputs will be saved to: c:\Users\sathv\OneDrive\Desktop\Sathvik education\5th Semester\PE\terramind_training_output

Initializing model...


In [ ]:
# Cell 9 — Training loop (with try/except for diagnostics)
print("\n" + "="*60)
print("🚀 STARTING TRAINING")
print("="*60)
print(f"Estimated time: ~{len(train_loader) * CONFIG['max_epochs'] // 60} minutes")
print("="*60 + "\n")

try:
    trainer.fit(
        model,
        train_dataloaders=train_loader,
        val_dataloaders=val_loader
    )

    print("\n" + "="*60)
    print("✅ TRAINING COMPLETED!")
    print("="*60)
    print(f"\nBest checkpoint: {checkpoint_callback.best_model_path}")
    print(f"Best val loss: {checkpoint_callback.best_model_score:.4f}")

except KeyboardInterrupt:
    print("\n⚠️ Training interrupted")
except Exception as e:
    print(f"\n❌ Training error: {e}")
    import traceback
    traceback.print_exc()



🚀 STARTING TRAINING
Estimated time: ~3 minutes




  | Name     | Type             | Params | Mode 
------------------------------------------------------
0 | backbone | TerraMindViT     | 87.3 M | train
1 | decoder  | Sequential       | 2.8 M  | train
2 | loss_fn  | CrossEntropyLoss | 0      | train
------------------------------------------------------
90.1 M    Trainable params
0         Non-trainable params
90.1 M    Total params
360.553   Total estimated model params size (MB)
182       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\sathv\AppData\Local\Programs\Python\Python311\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
c:\Users\sathv\AppData\Local\Programs\Python\Python311\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
INFO:lightning.pytorch.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

c:\Users\sathv\AppData\Local\Programs\Python\Python311\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Cell 10 — Post-train: load best model and evaluate 

from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn

if checkpoint_callback.best_model_path and os.path.exists(checkpoint_callback.best_model_path):
    print("\n🔄 Loading best model...")
    best_model = TerraMindFineTuner.load_from_checkpoint(
        checkpoint_callback.best_model_path, 
        config=CONFIG
    )
    best_model.eval()
    if torch.cuda.is_available():
        best_model = best_model.cuda()

    print("Evaluating on validation dataset (REGRESSION METRICS)...")

    l1_losses = []
    psnrs = []
    ssims = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating"):
            data, target = batch   # target must be clean image (not class mask)

            if torch.cuda.is_available():
                data = {k: v.cuda() for k, v in data.items()}
                target = target.cuda()

            pred = best_model(data)   # (B, C, H, W)

            # ---------- L1 ----------
            l1 = F.l1_loss(pred, target).item()
            l1_losses.append(l1)

            # ---------- PSNR + SSIM ----------
            pred_np = pred[0].detach().cpu().numpy().transpose(1,2,0)
            tgt_np  = target[0].detach().cpu().numpy().transpose(1,2,0)

            pred_np = np.clip(pred_np, 0.0, 1.0)
            tgt_np  = np.clip(tgt_np,  0.0, 1.0)

            # PSNR
            try:
                psnrs.append(psnr_fn(tgt_np, pred_np, data_range=1.0))
            except:
                pass

            # SSIM (band-wise average)
            ssim_vals = []
            for b in range(pred_np.shape[2]):
                try:
                    ssim_vals.append(
                        ssim_fn(
                            tgt_np[:,:,b],
                            pred_np[:,:,b],
                            data_range=1.0
                        )
                    )
                except:
                    pass
            if len(ssim_vals) > 0:
                ssims.append(np.mean(ssim_vals))

    print("\n================ REGRESSION RESULTS ================")
    print(f"Mean L1 loss : {np.mean(l1_losses):.6f}")
    print(f"Mean PSNR    : {np.mean(psnrs):.2f} dB")
    print(f"Mean SSIM    : {np.mean(ssims):.6f}")
    print("====================================================")

else:
    print("No checkpoint found, skipping evaluation.")

In [ ]:
# Cell 11 — Visualization utility (REGRESSION)

def visualize_predictions(model, dataloader, num_samples=4, output_path=OUTPUT_DIR / "predictions.png"):
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, num_samples * 4))
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    with torch.no_grad():
        for i, (data, target) in enumerate(dataloader):
            if i >= num_samples:
                break

            if torch.cuda.is_available():
                data = {k: v.cuda() for k, v in data.items()}
                target = target.cuda()

            pred = model(data)   # (B, C, H, W)

            first_modality = list(data.keys())[0]
            inp = data[first_modality][0].cpu().numpy()     # (C, H, W)
            tgt = target[0].cpu().numpy()
            out = pred[0].cpu().numpy()

            # --- RGB conversion (Sentinel-2: B4,B3,B2 → indices 2,1,0) ---
            def to_rgb(x):
                rgb = x[[2,1,0], :, :].transpose(1,2,0)
                rgb = np.clip((rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8), 0, 1)
                return rgb

            inp_rgb = to_rgb(inp)
            out_rgb = to_rgb(out)

            diff = np.mean(np.abs(out - inp), axis=0)

            axes[i, 0].imshow(inp_rgb)
            axes[i, 0].set_title("Input (TimeGate Aggregate)")
            axes[i, 0].axis("off")

            axes[i, 1].imshow(out_rgb)
            axes[i, 1].set_title("Output (Predicted Clean)")
            axes[i, 1].axis("off")

            im = axes[i, 2].imshow(diff, cmap="hot")
            axes[i, 2].set_title("Mean |Output − Input|")
            axes[i, 2].axis("off")
            plt.colorbar(im, ax=axes[i, 2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved visualization to: {output_path}")

In [ ]:
# Cell 12 — Save final model + config
final_dir = OUTPUT_DIR / 'final_model'
final_dir.mkdir(exist_ok=True)
if 'best_model' in globals():
    torch.save(best_model.state_dict(), final_dir / 'weights.pth')
else:
    torch.save(model.state_dict(), final_dir / 'weights.pth')

with open(final_dir / 'config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f"Final model saved to: {final_dir.absolute()}")


✓ Found 228 samples in D:\TerraMind_Dataset\train
✓ Found 57 samples in D:\TerraMind_Dataset\val
DataLoaders ready!
Train samples: 228
Val samples: 57
